# Evaluación — Trufi App Cochabamba
## CRISP-DM Fase 5 · Prueba, brecha, contraste y sensibilidad — iteración 2

Técnica adoptada en la Fase 4: ver `reports/03_modelado/adopcion.json`. Protocolo de esta fase declarado
antes de abrir la prueba: `reports/04_evaluacion/DECISIONES_04_evaluacion.md` (D-301 a D-312, versión 2).
La iteración 1 (prueba ya evaluada) está archivada en `reports/_iteracion1/04_evaluacion/`.

Orden de lectura: validación (Fase 4) → **prueba** (E2) → **brecha** (E3–E4) → **contraste** con GTFS (E5) →
diagnóstico espacial, análisis de residuos y calibración (E6, E6b, E7) → descripción (E8) → **sensibilidad** (E9) → criterio de éxito (E10).

In [1]:
import sys
from pathlib import Path


def _encontrar_raiz(inicio: Path) -> Path:
    for carpeta in [inicio, *inicio.parents]:
        if (carpeta / "pyproject.toml").exists():
            return carpeta
    raise RuntimeError("No se encontró pyproject.toml")


RAIZ = _encontrar_raiz(Path.cwd())
SRC_DIR = RAIZ / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

In [2]:
from trufi_ds.notebook_setup import *  # noqa: F403
from trufi_ds import modeling as md
from trufi_ds import preparation as prep
from trufi_ds.config import QUERIES_PARQUET, GTFS_DIR, PLAZA_14_SEPTIEMBRE

import json
import subprocess
import warnings
from datetime import datetime

import geopandas as gpd
import libpysal
import statsmodels.api as sm
from esda.moran import Moran, Moran_Local
from scipy.stats import mannwhitneyu, nbinom, spearmanr
from sklearn.model_selection import GroupKFold

SEMILLA = 42
np.random.seed(SEMILLA)
SUB = "04_evaluacion"
crear_directorios(SUB, f"{SUB}/figuras")
REPORTE = PROJECT_ROOT / "reports" / SUB
MODELADO = PROJECT_ROOT / "reports" / "03_modelado"
PROCESSED = PROJECT_ROOT / "data" / "processed"
INTERIM = PROJECT_ROOT / "data" / "interim"

adopcion = json.loads((MODELADO / "adopcion.json").read_text())
ADOPTADA, REFERENCIA = adopcion["adopted"], adopcion["reference_glm"]
PARAMS = {"M3": {"params": adopcion["m3_final_params"]}}
print(f"Técnica adoptada: {ADOPTADA} · referencia interpretable: {REFERENCIA}")

tabla = pl.read_parquet(PROCESSED / "tabla_minable.parquet").filter(pl.col("in_model")).sort("h3_cell")
area = gpd.read_file(PROJECT_ROOT / "reports/02_preparacion/area_estudio.geojson")
CENTRO = prep.area_center(area.geometry.iloc[0])  # D-022

Técnica adoptada: B1 · referencia interpretable: M2


## E1 · Reentrenar con todo el 80 % de entrenamiento

La técnica adoptada, B0 y la referencia interpretable se ajustan con todas las celdas de entrenamiento y la
configuración fijada en M8. Para separar las filas aún no se miran las de prueba: solo se usa la lista de bloques.

In [3]:
bloques_prueba = pl.read_csv(PROCESSED / "test_blocks.csv")["block_id"]
es_prueba = pl.col("block_id").is_in(bloques_prueba.implode())
entreno = tabla.filter(~es_prueba).to_pandas()

finales = {m: md.make(m, **PARAMS.get(m, {})).fit(entreno) for m in dict.fromkeys([ADOPTADA, "B0", "B1", REFERENCIA])}
config_final = {
    "adopted": ADOPTADA, "reference_glm": REFERENCIA, "trained_on": "80 % entrenamiento (bloques no reservados)",
    "n_train_cells": len(entreno), "b0_rate_per_inhabitant": finales["B0"].rate_,
    "b1": {"min_neighbors": finales["B1"].min_neighbors, "k_max": finales["B1"].k_max},
    "reference_params": {k: float(v) for k, v in finales[REFERENCIA].result_.params.items()},
    "m3_params": adopcion["m3_final_params"], "features": md.FEATURES, "seed": SEMILLA,
}
(REPORTE / "modelo_final_config.json").write_text(json.dumps(config_final, indent=2), encoding="utf-8")
config_final

{'adopted': 'B1',
 'reference_glm': 'M2',
 'trained_on': '80 % entrenamiento (bloques no reservados)',
 'n_train_cells': 1078,
 'b0_rate_per_inhabitant': np.float64(1.6391858971819295),
 'b1': {'min_neighbors': 3, 'k_max': 10},
 'reference_params': {'const': -5.900117308003525,
  'dist_centro_km': -0.1193740623858122,
  'log1p_pop_ring1': 1.0431063509619083,
  'log1p_pop_ring2': -0.29918804101070673,
  'alpha': 2.626936509090764},
 'm3_params': {'max_depth': None,
  'min_samples_leaf': 20,
  'learning_rate': 0.1},
 'features': ['dist_centro_km', 'log1p_pop_ring1', 'log1p_pop_ring2'],
 'seed': 42}

## E2 · Prueba (una sola vez, D-302)

Si `resultados_prueba.csv` ya existe, se lee y no se vuelve a evaluar.

In [4]:
RUTA_PRUEBA = REPORTE / "resultados_prueba.csv"
if RUTA_PRUEBA.exists():
    resultados_prueba = pd.read_csv(RUTA_PRUEBA)
    print(f"Prueba ya evaluada el {resultados_prueba['fecha'].iloc[0]} (commit {resultados_prueba['commit'].iloc[0][:8]}); no se repite.")
else:
    prueba = tabla.filter(es_prueba).to_pandas()
    commit = subprocess.run(["git", "rev-parse", "HEAD"], cwd=PROJECT_ROOT, capture_output=True, text=True).stdout.strip()
    fecha = datetime.now().isoformat(timespec="seconds")
    filas = []
    for m, modelo in finales.items():
        rol = "adoptada" if m == ADOPTADA else ("referencia interpretable" if m == REFERENCIA else "línea base")
        yhat = np.maximum(modelo.predict(prueba), md.EPS)
        filas.append({"model": m, "rol": rol, "distance_ring": "todas", **md.metrics(prueba["query_count"], yhat)})
        for a, g in prueba.assign(yhat=yhat).groupby("distance_ring"):
            filas.append({"model": m, "rol": rol, "distance_ring": a, **md.metrics(g["query_count"], g["yhat"])})
    resultados_prueba = pd.DataFrame(filas).assign(fecha=fecha, commit=commit, n_bloques=prueba["block_id"].nunique())
    resultados_prueba.to_csv(RUTA_PRUEBA, index=False)
resultados_prueba.round(4)

Prueba ya evaluada el 2026-09-27T13:39:37 (commit ac9ecc7d); no se repite.


,model,rol,distance_ring,poisson_deviance,d2,mae,rmse,calibration,spearman,n,fecha,commit,n_bloques
0,B1,adoptada,todas,981.7437,0.7647,892.5871,3799.5421,1.9623,0.8073,303,2026-09-27T13:39:37,ac9ecc7d16b51e927c71deb2b8a5bef515fa5069,12
1,B1,adoptada,A1,2617.6464,0.7159,2593.4024,7101.5372,2.1912,0.7702,82,2026-09-27T13:39:37,ac9ecc7d16b51e927c71deb2b8a5bef515fa5069,12
2,B1,adoptada,A2,700.6737,0.6297,496.9142,1441.1274,1.3098,0.9134,115,2026-09-27T13:39:37,ac9ecc7d16b51e927c71deb2b8a5bef515fa5069,12
3,B1,adoptada,A3,23.3100,-2.4215,2.5292,5.1027,0.8383,0.3969,48,2026-09-27T13:39:37,ac9ecc7d16b51e927c71deb2b8a5bef515fa5069,12
4,B1,adoptada,A4,19.3963,-0.2876,9.1096,17.0194,2.1260,0.7109,58,2026-09-27T13:39:37,ac9ecc7d16b51e927c71deb2b8a5bef515fa5069,12
5,B0,línea base,todas,1807.1684,0.5668,1043.8338,3495.8808,1.3718,0.7784,303,2026-09-27T13:39:37,ac9ecc7d16b51e927c71deb2b8a5bef515fa5069,12
6,B0,línea base,A1,3210.7128,0.6515,1706.5446,6451.4079,0.6223,0.7669,82,2026-09-27T13:39:37,ac9ecc7d16b51e927c71deb2b8a5bef515fa5069,12
7,B0,línea base,A2,1604.1438,0.1522,1087.0009,1448.0411,2.6454,0.8645,115,2026-09-27T13:39:37,ac9ecc7d16b51e927c71deb2b8a5bef515fa5069,12
8,B0,línea base,A3,789.8102,-114.9306,406.4011,630.7146,160.8955,0.6113,48,2026-09-27T13:39:37,ac9ecc7d16b51e927c71deb2b8a5bef515fa5069,12
9,B0,línea base,A4,1067.3473,-69.8546,548.8383,717.9332,122.9641,0.6802,58,2026-09-27T13:39:37,ac9ecc7d16b51e927c71deb2b8a5bef515fa5069,12


## E3 · Predicciones cruzadas de toda el área (R5)

`GroupKFold(5)` por `block_id` sobre **todas** las celdas del modelo (entrenamiento + prueba), con la configuración
ya fijada. Cada celda recibe la predicción de un modelo que no la vio: así un modelo flexible no borra la brecha.

In [5]:
def predicciones_cruzadas(df: pd.DataFrame, modelo: str, target: str = "query_count") -> tuple[np.ndarray, np.ndarray, pd.DataFrame]:
    yhat, pliegue, filas = np.empty(len(df)), np.empty(len(df), dtype=int), []
    for f, (tr, va) in enumerate(GroupKFold(5).split(df, groups=df["block_id"])):
        m = md.make(modelo, target=target, **PARAMS.get(modelo, {})).fit(df.iloc[tr])
        yhat[va] = np.maximum(m.predict(df.iloc[va]), md.EPS)
        pliegue[va] = f
        filas.append({"fold": f, **md.metrics(df.iloc[va][target], yhat[va])})
    return yhat, pliegue, pd.DataFrame(filas)


todas = tabla.to_pandas()
todas["in_test"] = todas["block_id"].isin(bloques_prueba.to_list())
yhat_adop, pliegue_todas, cv_todas = predicciones_cruzadas(todas, ADOPTADA)
yhat_ref, _, cv_todas_ref = predicciones_cruzadas(todas, REFERENCIA)
cv_todas.assign(model=ADOPTADA).to_csv(REPORTE / "cv_toda_el_area.csv", index=False)
cv_todas[["poisson_deviance", "d2", "spearman", "calibration"]].agg(["mean", "std"]).round(3)

,poisson_deviance,d2,spearman,calibration
mean,1128.773,0.720,0.844,1.172
std,805.346,0.196,0.042,0.411


## E4 · Residuos y puntaje de brecha (D-303)

In [6]:
def estimar_alpha(y: np.ndarray, yhat: np.ndarray) -> float:
    """alpha de una NB2 con solo intercepto y offset log(ŷ)."""
    modelo = sm.NegativeBinomial(y.astype(float), np.ones((len(y), 1)), loglike_method="nb2", offset=np.log(yhat))
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        for metodo in ("newton", "bfgs", "nm"):
            res = modelo.fit(start_params=[0.0, 1.0], method=metodo, maxiter=2000, disp=0)
            if res.mle_retvals.get("converged", False) and np.isfinite(res.params).all():
                break
    return float(res.params[-1])


def brecha(y: np.ndarray, yhat: np.ndarray, alpha: float | None = None) -> pd.DataFrame:
    y = np.asarray(y, float)
    alpha = estimar_alpha(y, yhat) if alpha is None else alpha
    n, p = 1 / alpha, 1 / (1 + alpha * yhat)
    p_low = nbinom.cdf(y, n, p)
    p_high = nbinom.sf(y - 1, n, p)
    residuo = (y - yhat) / np.sqrt(yhat + alpha * yhat**2)
    # D-303 (5 %) y D-310 (categoría descriptiva bajo_lo_esperado: p_low < 0,20 y residuo < 0)
    categoria = np.where(p_low < 0.05, "deficit",
                         np.where(p_high < 0.05, "exceso",
                                  np.where((p_low < 0.20) & (residuo < 0), "bajo_lo_esperado", "esperado")))
    return pd.DataFrame({"pearson_residual": residuo,
                         "p_low": p_low, "p_high": p_high, "gap_category": categoria, "alpha": alpha})


gap = brecha(todas["query_count"].to_numpy(), yhat_adop)
ALPHA = gap["alpha"].iloc[0]
pred = pd.concat([todas[["h3_cell", "block_id", "distance_ring", "lat", "lon", "population", "query_count", "user_count",
                         "dist_centro_km", "municipality", "pop_ring1", "pop_ring2", "dist_stop_m", "gtfs_covered", "route_count_500m",
                         "edge_cell", "in_test"]].reset_index(drop=True),
                  pd.DataFrame({"fold": pliegue_todas, "expected_count": yhat_adop, "expected_count_reference": yhat_ref}),
                  gap.drop(columns="alpha")], axis=1)
pred["model_id"] = ADOPTADA
pred.to_parquet(REPORTE / "predicciones_cruzadas.parquet", index=False)

reparto = pred["gap_category"].value_counts().rename_axis("gap_category").reset_index(name="celdas")
reparto["pct"] = (reparto["celdas"] / len(pred) * 100).round(2)
reparto.to_csv(REPORTE / "reparto_brecha.csv", index=False)
print(f"alpha (NB con offset log ŷ) = {ALPHA:.4f}")
reparto

alpha (NB con offset log ŷ) = 2.5970


,gap_category,celdas,pct
0,esperado,1253,90.73
1,exceso,105,7.60
2,bajo_lo_esperado,23,1.67


**Celdas de borde (riesgo 7).** Se comparan los residuos de las celdas a ≤ 1 km del borde del área con el resto.

In [7]:
borde = (pred.groupby("edge_cell").agg(celdas=("h3_cell", "size"), residuo_mediana=("pearson_residual", "median"),
                                       residuo_abs_medio=("pearson_residual", lambda s: s.abs().mean()),
                                       pct_deficit=("gap_category", lambda s: (s == "deficit").mean() * 100),
                                       pct_bajo_lo_esperado=("gap_category", lambda s: (s == "bajo_lo_esperado").mean() * 100),
                                       pct_exceso=("gap_category", lambda s: (s == "exceso").mean() * 100))
         .reset_index())
borde.to_csv(REPORTE / "residuos_borde.csv", index=False)
borde.round(3)

,edge_cell,celdas,residuo_mediana,residuo_abs_medio,pct_deficit,pct_bajo_lo_esperado,pct_exceso
0,False,1265,-0.337,42.734,0.0,1.502,7.905
1,True,116,-0.456,69.697,0.0,3.448,4.310


In [8]:
gdf = gpd.GeoDataFrame(pred, geometry=prep.cell_polygons(pred["h3_cell"].tolist()), crs="EPSG:4326")
lim = float(np.nanpercentile(np.abs(pred["pearson_residual"]), 98))
fig, ax = plt.subplots(figsize=(11, 9))
gdf.plot(ax=ax, column="pearson_residual", cmap="RdBu", vmin=-lim, vmax=lim, legend=True, edgecolor="none",
         legend_kwds={"label": "Residuo de Pearson NB (azul = más consultas de lo esperado)", "shrink": 0.7})
area.boundary.plot(ax=ax, color="#424242", linewidth=1)
ax.plot(CENTRO[1], CENTRO[0], "k*", ms=10)
ax.set_title(f"Brecha: residuo fuera de pliegue de {ADOPTADA} — N={len(pred):,} celdas; α={ALPHA:.3f}\n"
             f"déficit {(pred['gap_category'] == 'deficit').mean()*100:.1f} % · bajo lo esperado {(pred['gap_category'] == 'bajo_lo_esperado').mean()*100:.1f} % · exceso {(pred['gap_category'] == 'exceso').mean()*100:.1f} %")
ax.set_xlabel("Longitud"); ax.set_ylabel("Latitud"); fig.tight_layout()
guardar_figura(fig, "mapa_residuos", SUB); plt.close(fig)

## E5 · Contraste de la brecha con la cobertura GTFS (D-304)

Descriptivo: compara residuos entre celdas con y sin parada a ≤ 500 m. No es una prueba causal.

In [9]:
def contraste_cobertura(df: pd.DataFrame, umbral_m: float) -> dict:
    cubierta = df["dist_stop_m"] <= umbral_m
    r1, r0 = df.loc[cubierta, "pearson_residual"], df.loc[~cubierta, "pearson_residual"]
    u = mannwhitneyu(r1, r0, alternative="two-sided")
    return {"umbral_m": umbral_m, "celdas_cubiertas": int(cubierta.sum()), "celdas_no_cubiertas": int((~cubierta).sum()),
            "mediana_residuo_cubiertas": r1.median(), "mediana_residuo_no_cubiertas": r0.median(),
            "diferencia_medianas": r1.median() - r0.median(), "mann_whitney_U": u.statistic, "p_valor": u.pvalue,
            "cliff_delta": 2 * u.statistic / (len(r1) * len(r0)) - 1,
            "pct_deficit_cubiertas": (df.loc[cubierta, "gap_category"] == "deficit").mean() * 100,
            "pct_deficit_no_cubiertas": (df.loc[~cubierta, "gap_category"] == "deficit").mean() * 100,
            "pct_bajo_cubiertas": (df.loc[cubierta, "gap_category"] == "bajo_lo_esperado").mean() * 100,
            "pct_bajo_no_cubiertas": (df.loc[~cubierta, "gap_category"] == "bajo_lo_esperado").mean() * 100,
            "pct_exceso_cubiertas": (df.loc[cubierta, "gap_category"] == "exceso").mean() * 100,
            "pct_exceso_no_cubiertas": (df.loc[~cubierta, "gap_category"] == "exceso").mean() * 100}

brecha_cob = pd.DataFrame([contraste_cobertura(pred, u) for u in (400, 500, 750)])
brecha_cob.to_csv(REPORTE / "brecha_por_cobertura.csv", index=False)

fig, ax = plt.subplots(figsize=(7, 5))
grupos = [pred.loc[pred["gtfs_covered"] == g, "pearson_residual"] for g in (0, 1)]
ax.boxplot(grupos, showfliers=False, widths=0.5)
ax.set_xticks([1, 2], [f"No cubierta (n={len(grupos[0])})", f"Cubierta ≤ 500 m (n={len(grupos[1])})"])
ax.axhline(0, color="grey", lw=0.8)
fila500 = brecha_cob.set_index("umbral_m").loc[500]
ax.set_ylabel("Residuo de Pearson NB (fuera de pliegue)")
ax.set_title(f"Brecha por cobertura GTFS — Cliff δ = {fila500['cliff_delta']:+.3f}, p = {fila500['p_valor']:.2g}")
fig.tight_layout(); guardar_figura(fig, "residuos_por_cobertura", SUB); plt.close(fig)
brecha_cob.round(4)

,umbral_m,celdas_cubiertas,celdas_no_cubiertas,mediana_residuo_cubiertas,mediana_residuo_no_cubiertas,diferencia_medianas,mann_whitney_U,p_valor,cliff_delta,pct_deficit_cubiertas,pct_deficit_no_cubiertas,pct_bajo_cubiertas,pct_bajo_no_cubiertas,pct_exceso_cubiertas,pct_exceso_no_cubiertas
0,400,556,825,-0.1037,-0.4464,0.3427,314929.0,0.0,0.3731,0.0,0.0,1.0791,2.0606,11.1511,5.2121
1,500,617,764,-0.1264,-0.4552,0.3288,322723.0,0.0,0.3692,0.0,0.0,1.2966,1.9634,10.5348,5.2356
2,750,774,607,-0.2219,-0.4527,0.2308,299516.0,0.0,0.2750,0.0,0.0,1.6796,1.6474,9.0439,5.7661


## E6 · Diagnóstico espacial de los residuos (D-305)

In [10]:
celdas = pred["h3_cell"].tolist()
indice = {c: i for i, c in enumerate(celdas)}
vecinos = {i: [indice[n] for n in h3.grid_ring(c, 1) if n in indice] for i, c in enumerate(celdas)}
con_v = [i for i, v in vecinos.items() if v]
remap = {i: j for j, i in enumerate(con_v)}
w = libpysal.weights.W({remap[i]: [remap[n] for n in vecinos[i] if n in remap] for i in con_v}, silence_warnings=True)
w.transform = "r"
res_v = pred["pearson_residual"].to_numpy()[con_v]
np.random.seed(SEMILLA)
moran = Moran(res_v, w, permutations=999)
lisa = Moran_Local(res_v, w, permutations=999, seed=SEMILLA)
moran_res = pd.DataFrame([{"variable": "pearson_residual", "vecindad": "grid_ring k=1", "celdas": len(con_v),
                           "moran_I": moran.I, "EI": moran.EI, "z_sim": moran.z_sim, "p_sim": moran.p_sim,
                           **{f"lisa_{n}": int(((lisa.q == q) & (lisa.p_sim < 0.05)).sum()) for q, n in
                              [(1, "HH"), (2, "LH"), (3, "LL"), (4, "HL")]}}])
moran_res.to_csv(REPORTE / "moran_residuos.csv", index=False)

etiquetas = np.full(len(pred), "sin vecinos", dtype=object)
etiquetas[con_v] = np.where(lisa.p_sim < 0.05, pd.Series(lisa.q).map({1: "HH", 2: "LH", 3: "LL", 4: "HL"}).to_numpy(), "no significativo")
gdf["lisa"] = etiquetas
colores = {"HH": "#D32F2F", "LL": "#1976D2", "LH": "#90CAF9", "HL": "#FFAB91", "no significativo": "#EEEEEE", "sin vecinos": "white"}
fig, ax = plt.subplots(figsize=(11, 9))
for etiqueta, color in colores.items():
    sub = gdf[gdf["lisa"] == etiqueta]
    if len(sub):
        sub.plot(ax=ax, color=color, edgecolor="#BDBDBD", linewidth=0.1, label=f"{etiqueta} (n={len(sub)})")
area.boundary.plot(ax=ax, color="#424242", linewidth=1)
ax.legend(loc="upper right", fontsize=9)
ax.set_title(f"LISA del residuo de Pearson (vecindad H3 k=1) — I = {moran.I:.3f}, p = {moran.p_sim:.3f}")
ax.set_xlabel("Longitud"); ax.set_ylabel("Latitud"); fig.tight_layout()
guardar_figura(fig, "lisa_residuos", SUB); plt.close(fig)
moran_res.round(4)

,variable,vecindad,celdas,moran_I,EI,z_sim,p_sim,lisa_HH,lisa_LH,lisa_LL,lisa_HL
0,pearson_residual,grid_ring k=1,1370,0.129,-0.0007,8.1374,0.002,7,44,174,1


## E6b · Análisis de residuos de la técnica adoptada (D-312)

¿Queda estructura territorial que la técnica adoptada no captura? Si el residuo se relaciona con la distancia al
centro, la población o la población vecina, esa información podría mejorar la predicción. Si depende del anillo o
del municipio, hay zonas enteras sobre o subestimadas. El correlograma de Moran (k = 1, 2, 3) indica si los residuos
de celdas cercanas se parecen.

In [11]:
def moran_k(valores: np.ndarray, k: int) -> dict:
    vec = {i: [indice[n] for n in h3.grid_ring(c, k) if n in indice] for i, c in enumerate(celdas)}
    con = [i for i, v in vec.items() if v]
    rm = {i: j for j, i in enumerate(con)}
    w_k = libpysal.weights.W({rm[i]: [rm[n] for n in vec[i] if n in rm] for i in con}, silence_warnings=True)
    w_k.transform = "r"
    np.random.seed(SEMILLA)
    mk = Moran(valores[con], w_k, permutations=999)
    return {"moran_I": mk.I, "p_sim": mk.p_sim, "celdas": len(con)}

residuo = pred["pearson_residual"].to_numpy()
estructura = []
for var, x in [("dist_centro_km", pred["dist_centro_km"]), ("log_population", np.log(pred["population"])),
               ("log1p_pop_ring1", np.log1p(pred["pop_ring1"])), ("log1p_pop_ring2", np.log1p(pred["pop_ring2"]))]:
    rho = spearmanr(x, residuo)
    estructura.append({"analisis": "spearman_residuo_vs_variable", "grupo": var, "valor": rho.statistic, "p_valor": rho.pvalue, "celdas": len(pred)})
for k in (1, 2, 3):
    r = moran_k(residuo, k)
    estructura.append({"analisis": "moran_residuo", "grupo": f"grid_ring k={k}", "valor": r["moran_I"], "p_valor": r["p_sim"], "celdas": r["celdas"]})
for col in ("distance_ring", "municipality"):
    g = pred.groupby(col).agg(celdas=("h3_cell", "size"), mediana=("pearson_residual", "median"),
                              obs=("query_count", "sum"), esp=("expected_count", "sum"))
    for nombre, fila in g[g["celdas"] >= 10].iterrows():
        estructura.append({"analisis": f"residuo_por_{col}", "grupo": nombre, "valor": fila["mediana"],
                           "p_valor": np.nan, "celdas": int(fila["celdas"]), "ratio_obs_esp": fila["obs"] / fila["esp"]})
residuos_estructura = pd.DataFrame(estructura)
residuos_estructura.to_csv(REPORTE / "residuos_estructura.csv", index=False)

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
for ax, (var, x, etiqueta) in zip(axes, [("dist_centro_km", pred["dist_centro_km"], "Distancia al centro (km)"),
                                         ("log_population", np.log(pred["population"]), "log(población)"),
                                         ("log1p_pop_ring1", np.log1p(pred["pop_ring1"]), "log1p(población 1.ª corona)")]):
    bins = pd.qcut(x.rank(method="first"), 10, labels=False)
    agg = pd.DataFrame({"x": x, "r": residuo, "b": bins}).groupby("b").agg(x=("x", "median"), med=("r", "median"),
                                                                         q1=("r", lambda v: v.quantile(0.25)), q3=("r", lambda v: v.quantile(0.75)))
    ax.fill_between(agg["x"], agg["q1"], agg["q3"], color="#90CAF9", alpha=0.5, label="Q1–Q3")
    ax.plot(agg["x"], agg["med"], "o-", color="#1976D2", label="mediana")
    ax.axhline(0, color="grey", lw=0.8)
    rho = residuos_estructura.query("analisis == 'spearman_residuo_vs_variable' and grupo == @var")["valor"].iloc[0]
    ax.set_xlabel(etiqueta); ax.set_title(f"ρ de Spearman = {rho:+.3f}")
axes[0].set_ylabel("Residuo de Pearson NB (por decil)"); axes[0].legend()
fig.suptitle(f"Residuos de {ADOPTADA} frente a variables territoriales — N={len(pred):,} celdas")
fig.tight_layout(); guardar_figura(fig, "residuos_vs_variables", SUB); plt.close(fig)
residuos_estructura.round(4)

,analisis,grupo,valor,p_valor,celdas,ratio_obs_esp
0,spearman_residuo_vs_variable,dist_centro_km,-0.1314,0.000,1381,NaN
1,spearman_residuo_vs_variable,log_population,0.2477,0.000,1381,NaN
2,spearman_residuo_vs_variable,log1p_pop_ring1,0.2199,0.000,1381,NaN
3,spearman_residuo_vs_variable,log1p_pop_ring2,0.1720,0.000,1381,NaN
4,moran_residuo,grid_ring k=1,0.1290,0.002,1370,NaN
5,moran_residuo,grid_ring k=2,0.0175,0.057,1374,NaN
6,moran_residuo,grid_ring k=3,-0.0049,0.456,1379,NaN
7,residuo_por_distance_ring,A1,-0.2581,NaN,461,1.0241
8,residuo_por_distance_ring,A2,-0.3990,NaN,310,0.8526
9,residuo_por_distance_ring,A3,-0.2378,NaN,327,1.7071


## E7 · Calibración por deciles de ŷ y métricas por anillo

In [12]:
pred["decil"] = pd.qcut(pred["expected_count"].rank(method="first"), 10, labels=False) + 1
calib = (pred.groupby("decil").agg(celdas=("h3_cell", "size"), predicho=("expected_count", "sum"), observado=("query_count", "sum"),
                                   pred_medio=("expected_count", "mean"), obs_medio=("query_count", "mean"))
         .assign(ratio_obs_pred=lambda d: d["observado"] / d["predicho"]).reset_index())
calib.to_csv(REPORTE / "calibracion_deciles.csv", index=False)
metricas_anillo = pd.DataFrame([{"distance_ring": a, "model": ADOPTADA, **md.metrics(g["query_count"], g["expected_count"])}
                                for a, g in pred.groupby("distance_ring")]
                               + [{"distance_ring": a, "model": REFERENCIA, **md.metrics(g["query_count"], g["expected_count_reference"])}
                                  for a, g in pred.groupby("distance_ring")])
metricas_anillo.to_csv(REPORTE / "metricas_por_anillo.csv", index=False)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].loglog(calib["pred_medio"], calib["obs_medio"], "o-", color="#1976D2")
lo, hi = calib[["pred_medio", "obs_medio"]].min().min() * 0.8, calib[["pred_medio", "obs_medio"]].max().max() * 1.2
axes[0].plot([lo, hi], [lo, hi], "k--", lw=1)
axes[0].set_xlabel("Predicho medio por decil (log)"); axes[0].set_ylabel("Observado medio por decil (log)")
axes[0].set_title(f"Calibración por deciles de ŷ — {ADOPTADA}, N={len(pred):,}")
axes[1].bar(calib["decil"], calib["ratio_obs_pred"], color="#90CAF9")
axes[1].axhline(1, color="black", lw=1); axes[1].set_xlabel("Decil de ŷ"); axes[1].set_ylabel("Σ observado / Σ predicho")
axes[1].set_title("Razón observado/predicho por decil")
fig.tight_layout(); guardar_figura(fig, "calibracion", SUB); plt.close(fig)
calib.round(3)

,decil,celdas,predicho,observado,pred_medio,obs_medio,ratio_obs_pred
0,1,139,11.259,96,0.081,0.691,8.527
1,2,138,91.999,192,0.667,1.391,2.087
2,3,138,241.694,438,1.751,3.174,1.812
3,4,138,554.737,436,4.020,3.159,0.786
4,5,138,1194.909,1201,8.659,8.703,1.005
5,6,138,2837.083,2997,20.559,21.717,1.056
6,7,138,6991.971,9701,50.666,70.297,1.387
7,8,138,20437.788,48747,148.100,353.239,2.385
8,9,138,129759.428,168119,940.286,1218.254,1.296
9,10,138,1805004.270,1692409,13079.741,12263.833,0.938


## E8 · Descripción del modelo (D-307)

Las líneas base (B0–B1) no tienen coeficientes; B1 se describe por el tamaño de vecindad que usó. Las asociaciones con las variables
territoriales se describen con la referencia interpretable (NB2), ajustada con todo el entrenamiento. Son
**asociaciones del modelo**, no efectos causales.

In [13]:
ref = finales[REFERENCIA].result_
ic = ref.conf_int()
descripcion = pd.DataFrame({"termino": ref.params.index, "coef": ref.params.values, "ic95_inf": ic[0].values,
                            "ic95_sup": ic[1].values, "p_valor": ref.pvalues.values})
es_coef = ~descripcion["termino"].isin(["alpha"])
for c in ["coef", "ic95_inf", "ic95_sup"]:
    descripcion.loc[es_coef, c.replace("coef", "irr").replace("ic95", "irr_ic95")] = np.exp(descripcion.loc[es_coef, c])
descripcion["modelo"] = REFERENCIA
descripcion.to_csv(REPORTE / "descripcion_modelo.csv", index=False)

finales["B1"].predict(todas)  # B1 entrenado con el 80 %; registra el k usado por celda
b1_k = pd.Series(finales["B1"].k_used_).value_counts().sort_index().rename_axis("k").reset_index(name="celdas")
b1_k.to_csv(REPORTE / "b1_tamano_vecindad.csv", index=False)

coefs = descripcion[descripcion["termino"].isin(md.FEATURES)]
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.errorbar(coefs["irr"], range(len(coefs)), xerr=[coefs["irr"] - coefs["irr_ic95_inf"], coefs["irr_ic95_sup"] - coefs["irr"]],
            fmt="o", color="#388E3C", capsize=4)
ax.axvline(1, color="grey", ls="--"); ax.set_yticks(range(len(coefs)), coefs["termino"])
ax.set_xlabel("IRR = exp(β) con IC 95 %")
ax.set_title(f"Asociaciones de la referencia {REFERENCIA} (offset log población) — N={len(entreno):,}")
fig.tight_layout(); guardar_figura(fig, "descripcion_modelo_irr", SUB); plt.close(fig)
descripcion.round(4)

,termino,coef,ic95_inf,ic95_sup,p_valor,irr,irr_ic95_inf,irr_ic95_sup,modelo
0,const,-5.9001,-6.6501,-5.1502,0.0000,0.0027,0.0013,0.0058,M2
1,dist_centro_km,-0.1194,-0.1355,-0.1032,0.0000,0.8875,0.8733,0.9019,M2
2,log1p_pop_ring1,1.0431,0.8138,1.2724,0.0000,2.8380,2.2566,3.5693,M2
3,log1p_pop_ring2,-0.2992,-0.5229,-0.0754,0.0088,0.7414,0.5928,0.9273,M2
4,alpha,2.6269,2.3867,2.8672,0.0000,NaN,NaN,NaN,M2


## E9 · Sensibilidad (D-306)

Cada variación reconstruye la tabla con el mismo código de la Etapa 2 (`trufi_ds.preparation`) y la misma área,
repite las predicciones cruzadas de E3 con la técnica adoptada y compara el ranking de brecha con el de referencia.

In [14]:
area_geom = area.geometry.iloc[0]
kontur_2023 = pl.read_parquet(INTERIM / "kontur_2023_h3r8.parquet")
kontur_2022 = pl.read_parquet(INTERIM / "kontur_2022_h3r8.parquet")
consultas_limpias = pl.read_parquet(INTERIM / "queries_limpias.parquet")
crudas = pl.read_parquet(QUERIES_PARQUET)
anomalos = pl.read_csv(PROJECT_ROOT / "reports/01_data_understanding/usuarios_anomalos.csv").filter(pl.col("flag_anomalo"))["userID"]

def tabla_variante(consultas: pl.DataFrame, kontur: pl.DataFrame, pop_min: int = 10,
                   centro: tuple[float, float] = CENTRO) -> pd.DataFrame:
    t = prep.build_mining_table(consultas, kontur, area_geom, GTFS_DIR, center=centro, pop_min=pop_min)
    return t.filter(pl.col("in_model")).sort("h3_cell").to_pandas()

# Control: la tabla de referencia se reproduce con el mismo código
control = tabla_variante(consultas_limpias, kontur_2023)
assert control["h3_cell"].tolist() == todas["h3_cell"].tolist() and (control["query_count"].to_numpy() == todas["query_count"].to_numpy()).all()

variantes = {
    "kontur_2022": (tabla_variante(consultas_limpias, kontur_2022), "query_count"),
    **{f"distancia_{km}km": (tabla_variante(prep.clean_queries(crudas, area_geom, anomalos, km * 1000)[0], kontur_2023), "query_count")
       for km in (20, 30, 40)},
    "centro_plaza_14_septiembre": (tabla_variante(consultas_limpias, kontur_2023,
                                                  centro=(PLAZA_14_SEPTIEMBRE["lat"], PLAZA_14_SEPTIEMBRE["lon"])), "query_count"),
    "poblacion_min_50": (tabla_variante(consultas_limpias, kontur_2023, pop_min=50), "query_count"),
    "objetivo_user_count": (todas.drop(columns="in_test"), "user_count"),
}
base_rank = pred.set_index("h3_cell")["pearson_residual"]
sens = [{"variante": "referencia", "celdas": len(pred), "deviance_mean": cv_todas["poisson_deviance"].mean(),
         "deviance_std": cv_todas["poisson_deviance"].std(ddof=1), "d2_mean": cv_todas["d2"].mean(), "d2_std": cv_todas["d2"].std(ddof=1),
         "alpha": ALPHA, "pct_deficit": (pred["gap_category"] == "deficit").mean() * 100,
         "pct_bajo_lo_esperado": (pred["gap_category"] == "bajo_lo_esperado").mean() * 100,
         "celdas_comunes": len(pred), "spearman_brecha_vs_referencia": 1.0}]
for nombre, (df_v, objetivo) in variantes.items():
    yhat_v, _, cv_v = predicciones_cruzadas(df_v, ADOPTADA, target=objetivo)
    gap_v = brecha(df_v[objetivo].to_numpy(), yhat_v)
    r_v = pd.Series(gap_v["pearson_residual"].to_numpy(), index=df_v["h3_cell"])
    comunes = base_rank.index.intersection(r_v.index)
    sens.append({"variante": nombre, "celdas": len(df_v), "deviance_mean": cv_v["poisson_deviance"].mean(),
                 "deviance_std": cv_v["poisson_deviance"].std(ddof=1), "d2_mean": cv_v["d2"].mean(), "d2_std": cv_v["d2"].std(ddof=1),
                 "alpha": gap_v["alpha"].iloc[0], "pct_deficit": (gap_v["gap_category"] == "deficit").mean() * 100,
                 "pct_bajo_lo_esperado": (gap_v["gap_category"] == "bajo_lo_esperado").mean() * 100,
                 "celdas_comunes": len(comunes),
                 "spearman_brecha_vs_referencia": spearmanr(base_rank[comunes], r_v[comunes]).statistic})
for u in (400, 750):
    fila = brecha_cob.set_index("umbral_m").loc[u]
    sens.append({"variante": f"cobertura_{u}m (solo E5)", "celdas": len(pred), "celdas_comunes": len(pred),
                 "spearman_brecha_vs_referencia": np.nan,
                 "nota": f"Cliff δ = {fila['cliff_delta']:+.3f}; déficit cubiertas {fila['pct_deficit_cubiertas']:.1f} % vs no cubiertas {fila['pct_deficit_no_cubiertas']:.1f} %"})
sensibilidad = pd.DataFrame(sens)
sensibilidad.to_csv(REPORTE / "sensibilidad.csv", index=False)
sensibilidad.round(4)

,variante,celdas,deviance_mean,deviance_std,d2_mean,d2_std,alpha,pct_deficit,pct_bajo_lo_esperado,celdas_comunes,spearman_brecha_vs_referencia,nota
0,referencia,1381,1128.7729,805.3463,0.7204,0.1963,2.5970,0.0,1.6655,1381,1.0000,NaN
1,kontur_2022,1382,1246.6057,701.5752,0.6890,0.2143,2.6293,0.0,2.1708,1354,0.8850,NaN
2,distancia_20km,1376,1114.9498,808.5210,0.7239,0.1918,2.5511,0.0,1.9622,1376,0.8672,NaN
3,distancia_30km,1380,1125.7932,802.5717,0.7201,0.1976,2.6302,0.0,1.8116,1380,0.9240,NaN
4,distancia_40km,1381,1127.8888,805.2301,0.7205,0.1962,2.5852,0.0,1.8827,1381,0.9966,NaN
5,centro_plaza_14_septiembre,1381,1128.7729,805.3463,0.7204,0.1963,2.5970,0.0,1.6655,1381,1.0000,NaN
6,poblacion_min_50,1158,1464.2522,1035.4171,0.7169,0.1243,2.2438,0.0,4.7496,1158,0.8835,NaN
7,objetivo_user_count,1381,316.8526,199.7118,0.7339,0.1911,2.0215,0.0,2.6068,1381,0.9706,NaN
8,cobertura_400m (solo E5),1381,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1381,NaN,Cliff δ = +0.373; déficit cubiertas 0.0 % vs n...
9,cobertura_750m (solo E5),1381,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1381,NaN,Cliff δ = +0.275; déficit cubiertas 0.0 % vs n...


## E10 · Criterio de éxito (D-301)

In [15]:
rp = resultados_prueba[resultados_prueba["distance_ring"] == "todas"].set_index("model")
dev_adop, dev_b0 = rp.loc[ADOPTADA, "poisson_deviance"], rp.loc["B0", "poisson_deviance"]
criterio1 = dev_adop < dev_b0
spearman_min = sensibilidad["spearman_brecha_vs_referencia"].dropna().min()
peor = sensibilidad.loc[sensibilidad["spearman_brecha_vs_referencia"].idxmin(), "variante"]
criterio2 = spearman_min >= 0.7

filas_prueba = "\n".join(
    f"| {m} | {r['rol']} | {r['poisson_deviance']:.1f} | {r['d2']:.3f} | {r['mae']:.1f} | {r['calibration']:.3f} | {r['spearman']:.3f} |"
    for m, r in rp.iterrows())
filas_sens = "\n".join(
    f"| {r.variante} | {r.celdas} | " + ("—" if pd.isna(r.spearman_brecha_vs_referencia) else f"{r.spearman_brecha_vs_referencia:.3f}")
    + " | " + ("—" if pd.isna(r.pct_deficit) else f"{r.pct_deficit:.1f} %") + " |"
    for r in sensibilidad.itertuples())
fila500 = brecha_cob.set_index("umbral_m").loc[500]
criterio_md = f"""# Criterio de éxito (E10)

Generado por `notebooks/04_evaluacion.ipynb`. Criterio declarado en D-301 antes de abrir la prueba.

| # | Criterio | Valor | Cumple |
|---|---|---|---|
| 1 | Devianza de prueba de {ADOPTADA} < devianza de prueba de B0 | {dev_adop:.1f} vs {dev_b0:.1f} | {"sí" if criterio1 else "no"} |
| 2 | Spearman del ranking de brecha ≥ 0,7 en todas las variaciones de E9 | mínimo {spearman_min:.3f} ({peor}) | {"sí" if criterio2 else "no"} |

**Resultado:** {"se cumplen ambos criterios." if criterio1 and criterio2 else ("se cumple el criterio 1; el ranking de brecha es inestable (riesgo 10): el mapa se entrega con esa advertencia." if criterio1 else "no se cumple el criterio 1: la técnica adoptada no supera a B0 en prueba.")}

## Prueba (única evaluación, {resultados_prueba['fecha'].iloc[0]}, commit `{resultados_prueba['commit'].iloc[0][:8]}`, {int(resultados_prueba['n_bloques'].iloc[0])} bloques)

| Técnica | Rol | Devianza | D² | MAE | Calibración | Spearman |
|---|---|---|---|---|---|---|
{filas_prueba}

Validación cruzada espacial de la Fase 4 (media ± DE): ver `reports/03_modelado/resumen_cv.csv`. La prueba contiene
el {100 * tabla.filter(es_prueba)['query_count'].sum() / tabla['query_count'].sum():.1f} % de las consultas en {int(resultados_prueba['n_bloques'].iloc[0])} bloques: léase con prudencia.

## Brecha y contraste

- α (NB con offset log ŷ) = {ALPHA:.3f}; déficit {(pred['gap_category'] == 'deficit').mean()*100:.1f} %, bajo lo esperado (D-310, descriptivo) {(pred['gap_category'] == 'bajo_lo_esperado').mean()*100:.1f} %, exceso {(pred['gap_category'] == 'exceso').mean()*100:.1f} % de {len(pred):,} celdas.
- Cobertura (≤ 500 m): mediana del residuo {fila500['mediana_residuo_cubiertas']:+.3f} en cubiertas vs {fila500['mediana_residuo_no_cubiertas']:+.3f} en no cubiertas; Cliff δ = {fila500['cliff_delta']:+.3f} (p = {fila500['p_valor']:.2g}). Déficit: {fila500['pct_deficit_cubiertas']:.1f} % vs {fila500['pct_deficit_no_cubiertas']:.1f} %. Asociación descriptiva, no causal.
- Moran de residuos (H3 k=1): I = {moran.I:.3f}, p = {moran.p_sim:.3f}.

## Sensibilidad (`sensibilidad.csv`)

| Variación | Celdas | Spearman vs referencia | % déficit |
|---|---|---|---|
{filas_sens}
"""
(REPORTE / "criterio_exito.md").write_text(criterio_md, encoding="utf-8")
print(criterio_md)

# Criterio de éxito (E10)

Generado por `notebooks/04_evaluacion.ipynb`. Criterio declarado en D-301 antes de abrir la prueba.

| # | Criterio | Valor | Cumple |
|---|---|---|---|
| 1 | Devianza de prueba de B1 < devianza de prueba de B0 | 981.7 vs 1807.2 | sí |
| 2 | Spearman del ranking de brecha ≥ 0,7 en todas las variaciones de E9 | mínimo 0.867 (distancia_20km) | sí |

**Resultado:** se cumplen ambos criterios.

## Prueba (única evaluación, 2026-09-27T13:39:37, commit `ac9ecc7d`, 12 bloques)

| Técnica | Rol | Devianza | D² | MAE | Calibración | Spearman |
|---|---|---|---|---|---|---|
| B1 | adoptada | 981.7 | 0.765 | 892.6 | 1.962 | 0.807 |
| B0 | línea base | 1807.2 | 0.567 | 1043.8 | 1.372 | 0.778 |
| M2 | referencia interpretable | 1210.0 | 0.710 | 562.4 | 0.372 | 0.815 |

Validación cruzada espacial de la Fase 4 (media ± DE): ver `reports/03_modelado/resumen_cv.csv`. La prueba contiene
el 11.8 % de las consultas en 12 bloques: léase con prudencia.

## Brecha y contraste

- α